# SOmics reproducibility notebook (cleaned copy)

This is an archived, minimally cleaned copy of the original research notebook. Original analysis code has been preserved in its existing order. Cell outputs, exploratory troubleshooting cells, and unrelated Drive file-search cells have been removed.

**Not yet verified as a standalone run:** This notebook requires access to original TCGA and Visium inputs, a Google Colab/Drive configuration, and packages compatible with the original environment. See the accompanying README before execution.

**Important:** Results can depend on external input files and execution order. This copy has not been re-executed or validated end-to-end.


In [ ]:
# =========================
# SOmics-ML (FINAL) — TCGA bulk training → Visium spot prediction
# + LOSO robustness
# + gene-set scoring (median + frac above threshold per sample)
# + model-derived hub genes (RF importance + LR nonzero), recurrent ≥2 samples/type
# =========================

# ---------- 0) Setup ----------
from google.colab import drive
drive.mount("/content/drive")

import os, glob, gzip, json
import numpy as np
import pandas as pd

from scipy import sparse
from scipy.io import mmread
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, auc

import matplotlib.pyplot as plt


# ---------- 1) Paths + constants (match your Drive setup) ----------
DRIVE_ROOT = "/content/drive/MyDrive"

TRAIN_DIR   = f"{DRIVE_ROOT}/ML Train Data"
SAMPLES_DIR = f"{DRIVE_ROOT}/Samples"
OUT_DIR     = f"{DRIVE_ROOT}/ML_ExternalVal_FINAL"

os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_CSV = os.path.join(TRAIN_DIR, "TCGA_OV_CAF_Immune_training.csv")

SAMPLES = [f"Sample_{i}" for i in range(1, 9)]
TYPE_MAP = {f"Sample_{i}": ("OCS" if i <= 4 else "HGSC") for i in range(1, 9)}

THR = 0.5                 # binary pCAF threshold
TOP_VAR_GENES = 1000      # fixed feature space size
RF_N_ITER = 15            # randomized search iterations
RF_N_EST = 300            # n_estimators in RF base model

print("TRAIN_CSV:", TRAIN_CSV)
print("SAMPLES_DIR:", SAMPLES_DIR)
print("OUT_DIR:", OUT_DIR)
print("TYPE_MAP:", TYPE_MAP)


# ---------- 2) Gene sets (explicit lists for Supplement) ----------
# These are the sets you outlined (checkpoint/exhaustion vs stromal exclusion separated).
GENESETS = {
    "checkpoint_exhaustion": [
        "PDCD1","CD274","PDCD1LG2","CTLA4","LAG3","TIGIT","HAVCR2","ENTPD1","NT5E",
        "FOXP3","IL2RA","IKZF2"
    ],
    "stromal_caf_exclusion": ["FAP","ACTA2","COL1A1","VEGFA","FLT1","ANGPT2"],
    "myeloid_suppressive": ["CD68","CD163","MRC1","CSF1R","CD14","ITGAM","IL10","TGFB1","S100A8","S100A9","NOS2"],
    "cytotoxic_nk": ["NCAM1","KLRK1","NCR1","GZMB","PRF1","GZMA","IFNG","TNF"],
    "antigen_bcell": [
        "HLA-DRA","HLA-DRB1","HLA-A","HLA-B","TAP1","B2M",
        "MS4A1","CD19","CD79A","CD79B","IGKC","IGHG1","IGHG2","IGHG3","MZB1","XBP1","JCHAIN",
        "CXCL13","CXCR5","CCL19","CCL21","CXCL9","CXCL10"
    ],
}

# Save gene sets to a single CSV for Supplement
gs_rows = []
for set_name, genes in GENESETS.items():
    for g in genes:
        gs_rows.append({"geneset": set_name, "gene": g})
pd.DataFrame(gs_rows).to_csv(os.path.join(OUT_DIR, "Supplement_GeneSets_ExactLists.csv"), index=False)


# ---------- 3) TCGA load + preprocessing ----------
def strip_ensembl_version(s: str) -> str:
    return s.split(".")[0] if isinstance(s, str) else s

df = pd.read_csv(TRAIN_CSV, index_col=0)
print("Raw training df shape:", df.shape)

meta_cols = [c for c in ["CAF_score", "Immune_score", "Label"] if c in df.columns]
gene_cols = [c for c in df.columns if c not in meta_cols]

# standardize gene columns (remove Ensembl version suffixes)
gene_cols_stripped = [strip_ensembl_version(c) for c in gene_cols]
Xgenes = df[gene_cols].copy()
Xgenes.columns = gene_cols_stripped
Xgenes = Xgenes.loc[:, ~Xgenes.columns.duplicated()]  # drop duplicate after stripping

df_std = pd.concat([df[meta_cols], Xgenes], axis=1) if meta_cols else Xgenes
print("Standardized df shape:", df_std.shape)

# labels
if "Label" in df_std.columns:
    y = (df_std["Label"].astype(str).str.lower() == "caf_high").astype(int)
else:
    # derive labels as in your Methods (top quartile CAF_score or Immune_score; exclude others)
    if ("CAF_score" not in df_std.columns) or ("Immune_score" not in df_std.columns):
        raise ValueError("Need Label or CAF_score + Immune_score to define labels.")
    caf_q = df_std["CAF_score"].quantile(0.75)
    imm_q = df_std["Immune_score"].quantile(0.75)

    caf_high = df_std["CAF_score"] >= caf_q
    imm_high = df_std["Immune_score"] >= imm_q

    keep = (caf_high | imm_high) & ~(caf_high & imm_high)
    df_std = df_std.loc[keep].copy()
    caf_high = caf_high.loc[keep]

    y = caf_high.astype(int)
    df_std["Label"] = np.where(caf_high, "CAF_high", "Immune_high")

print("Label counts:\n", pd.Series(y).value_counts())

# feature matrix
gene_cols_std = [c for c in df_std.columns if c not in ["CAF_score", "Immune_score", "Label"]]
X = df_std[gene_cols_std].astype(float).copy()
print("X shape:", X.shape, "y shape:", y.shape)


# ---------- 4) Train/test split + variance feature selection (train-only) ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

gene_variances = X_train.var(axis=0)
top_k = min(TOP_VAR_GENES, gene_variances.shape[0])
model_features = gene_variances.sort_values().tail(top_k).index.tolist()

X_train_red = X_train[model_features]
X_test_red  = X_test[model_features]

FEATURES_TXT = os.path.join(OUT_DIR, "model_features_top1000.txt")
with open(FEATURES_TXT, "w") as f:
    f.write("\n".join(model_features))

print("Reduced feature shapes:", X_train_red.shape, X_test_red.shape)
print("Saved feature list:", FEATURES_TXT)


# ---------- 5) 10-fold CV randomized search for RF ----------
cv10 = KFold(n_splits=10, shuffle=True, random_state=42)

rf_base = RandomForestClassifier(
    n_estimators=RF_N_EST,
    random_state=42,
    n_jobs=-1
)

rf_param_dist = {
    "max_depth": [3, 4, 5, 6],
    "min_samples_split": [10, 20, 30],
    "min_samples_leaf": [5, 10, 15],
    "max_features": [0.2, 0.3, "sqrt"]
}

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=RF_N_ITER,
    scoring="roc_auc",
    cv=cv10,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train_red, y_train)
rf = rf_search.best_estimator_

rf_test_p = rf.predict_proba(X_test_red)[:, 1]
rf_test_auc = roc_auc_score(y_test, rf_test_p)
rf_test_acc = accuracy_score(y_test, (rf_test_p >= THR).astype(int))

print("\nRF best params:", rf_search.best_params_)
print("RF CV AUC (best):", rf_search.best_score_)
print("RF Test AUC:", rf_test_auc, "Test Acc:", rf_test_acc)


# ---------- 6) Logistic Regression (L1, interpretable) ----------
lr = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=5000,
    C=1.0,
    n_jobs=-1
)
lr.fit(X_train_red, y_train)

lr_test_p = lr.predict_proba(X_test_red)[:, 1]
lr_test_auc = roc_auc_score(y_test, lr_test_p)
lr_test_acc = accuracy_score(y_test, (lr_test_p >= THR).astype(int))

print("\nLR Test AUC:", lr_test_auc, "Test Acc:", lr_test_acc)


# ---------- 7) Save metrics/params ----------
params_out = {
    "rf_best_params": rf_search.best_params_,
    "rf_cv_auc_best": float(rf_search.best_score_),
    "rf_test_auc": float(rf_test_auc),
    "rf_test_acc": float(rf_test_acc),
    "lr_params": {"penalty":"l1","solver":"saga","C":1.0,"max_iter":5000},
    "lr_test_auc": float(lr_test_auc),
    "lr_test_acc": float(lr_test_acc),
    "threshold": THR,
    "n_features": int(len(model_features))
}
with open(os.path.join(OUT_DIR, "trained_model_params.json"), "w") as f:
    json.dump(params_out, f, indent=2)

print("Saved trained_model_params.json")


# ---------- 8) Optional ROC plot ----------
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_test_p)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_test_p)

plt.figure(figsize=(6,6))
plt.plot(fpr_rf, tpr_rf, linewidth=2, label=f"RF (AUC={auc(fpr_rf,tpr_rf):.2f})")
plt.plot(fpr_lr, tpr_lr, linewidth=2, label=f"LR-L1 (AUC={auc(fpr_lr,tpr_lr):.2f})")
plt.plot([0,1],[0,1],'k--', alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Held-out TCGA test set ROC")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ROC_RF_vs_LR.png"), dpi=300)
plt.show()


# ---------- 9) Model-derived hub candidates (Stage A of 2.11.7) ----------
lr_coef = pd.Series(lr.coef_.ravel(), index=model_features)
rf_imp  = pd.Series(rf.feature_importances_, index=model_features).sort_values(ascending=False)

lr_nonzero = set(lr_coef[lr_coef != 0].index)
rf_top = set(rf_imp.head(200).index)  # generous; recurrence filtering later

model_hub_candidates = sorted(lr_nonzero.union(rf_top))

pd.Series(lr_coef).sort_values(ascending=False).head(100).to_csv(os.path.join(OUT_DIR, "LR_global_hubs_pos_CAF_high.csv"))
pd.Series(lr_coef).sort_values(ascending=True).head(100).to_csv(os.path.join(OUT_DIR, "LR_global_hubs_neg_Immune_high.csv"))
rf_imp.head(200).to_csv(os.path.join(OUT_DIR, "RF_global_feature_importance_top200.csv"))

with open(os.path.join(OUT_DIR, "ModelHubCandidates_union_RF_LR.txt"), "w") as f:
    f.write("\n".join(model_hub_candidates))

print("LR nonzero:", len(lr_nonzero), "RF top200:", len(rf_top), "Union hub candidates:", len(model_hub_candidates))


# ---------- 10) Visium I/O helpers ----------
def _read_tsv_maybe_gz(path_no_gz):
    if os.path.exists(path_no_gz):
        return pd.read_csv(path_no_gz, sep="\t", header=None)
    if os.path.exists(path_no_gz + ".gz"):
        return pd.read_csv(path_no_gz + ".gz", sep="\t", header=None, compression="gzip")
    raise FileNotFoundError(f"Missing {path_no_gz} or {path_no_gz}.gz")

def _read_mtx_maybe_gz(path_no_gz):
    if os.path.exists(path_no_gz):
        return mmread(path_no_gz)
    if os.path.exists(path_no_gz + ".gz"):
        with gzip.open(path_no_gz + ".gz", "rb") as f:
            return mmread(f)
    raise FileNotFoundError(f"Missing {path_no_gz} or {path_no_gz}.gz")

def read_visium_mtx(sample_dir):
    """
    Returns:
      counts: genes x spots sparse
      ens_ids: Ensembl IDs (features.tsv col 1)
      gene_symbols: gene symbols (features.tsv col 2)
      spot_ids: barcodes
    """
    mtx_dir = os.path.join(sample_dir, "filtered_feature_bc_matrix")
    if not os.path.isdir(mtx_dir):
        raise FileNotFoundError(f"Missing filtered_feature_bc_matrix in {sample_dir}")

    m = _read_mtx_maybe_gz(os.path.join(mtx_dir, "matrix.mtx")).tocsr()

    feats = _read_tsv_maybe_gz(os.path.join(mtx_dir, "features.tsv"))
    ens_ids = feats.iloc[:, 0].astype(str).tolist()        # Ensembl
    gene_symbols = feats.iloc[:, 1].astype(str).tolist()   # symbol

    bcs = _read_tsv_maybe_gz(os.path.join(mtx_dir, "barcodes.tsv"))
    spot_ids = bcs.iloc[:, 0].astype(str).tolist()

    return m, ens_ids, gene_symbols, spot_ids

def read_tissue_positions(sample_dir):
    spatial_dir = os.path.join(sample_dir, "spatial")
    f = os.path.join(spatial_dir, "tissue_positions_list.csv")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing tissue_positions_list.csv in {spatial_dir}")

    pos = pd.read_csv(f, header=None)
    pos.columns = ["barcode","in_tissue","array_row","array_col","pxl_row","pxl_col"]
    return pos

def log1p_cpm_sparse(counts_csr):
    counts = counts_csr.tocsc(copy=True)
    col_sums = np.array(counts.sum(axis=0)).ravel()
    col_sums[col_sums == 0] = 1.0
    inv = 1e6 / col_sums
    counts = counts.multiply(inv)
    counts = counts.tocsr()
    counts.data = np.log1p(counts.data)
    return counts

def align_to_model_features(norm_csr, visium_ens_ids, model_features):
    """
    norm_csr: genes x spots (same order as visium_ens_ids)
    model_features: TCGA features (Ensembl IDs, no version suffix)
    """
    visium_clean = [g.split(".")[0] for g in visium_ens_ids]
    model_clean  = [g.split(".")[0] for g in model_features]

    gene_to_idx = {}
    for i,g in enumerate(visium_clean):
        if g not in gene_to_idx:
            gene_to_idx[g] = i

    present_idx = []
    present_mask = []
    for g in model_clean:
        if g in gene_to_idx:
            present_idx.append(gene_to_idx[g])
            present_mask.append(True)
        else:
            present_mask.append(False)

    if sum(present_mask) == 0:
        return None, 0.0

    sub = norm_csr[present_idx, :]   # genes_present x spots
    X = sub.T.tocsr()                # spots x genes_present

    if all(present_mask):
        return X, 1.0

    n_spots = X.shape[0]
    cols = []
    j = 0
    for p in present_mask:
        if p:
            cols.append(X[:, j]); j += 1
        else:
            cols.append(sparse.csr_matrix((n_spots, 1)))
    Xfull = sparse.hstack(cols).tocsr()
    return Xfull, (sum(present_mask) / len(present_mask))

# Map Ensembl IDs to gene symbols (for readable hub outputs)
def build_ens_to_symbol_map(sample_dir):
    _, ens_ids, gene_symbols, _ = read_visium_mtx(sample_dir)
    ens_clean = [e.split(".")[0] for e in ens_ids]
    sym_clean = [s.split(".")[0] for s in gene_symbols]
    return dict(zip(ens_clean, sym_clean))


# ---------- 11) Gene-set scoring helpers (Visium) ----------
def score_geneset(norm_csr, gene_symbols, genes):
    """
    gene_symbols: Visium symbols (features.tsv col 2)
    genes: list of symbols for the geneset
    """
    sym_clean = [g.split(".")[0] for g in gene_symbols]
    gene_to_idx = {}
    for i,g in enumerate(sym_clean):
        if g not in gene_to_idx:
            gene_to_idx[g] = i

    idx = [gene_to_idx[g] for g in genes if g in gene_to_idx]
    if len(idx) == 0:
        return np.full(norm_csr.shape[1], np.nan)

    sub = norm_csr[idx, :]  # genes x spots
    return np.array(sub.mean(axis=0)).ravel()

def frac_above_threshold(v, thr):
    v = np.asarray(v)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return np.nan
    return float((v >= thr).mean())


# ---------- 12) External spot-level inference + gene set scores ----------
all_spots = []
module_rows = []  # per-sample gene-set summary

for s in SAMPLES:
    sd = os.path.join(SAMPLES_DIR, s)
    print("Processing:", s)

    counts, ens_ids, gene_symbols, spot_ids = read_visium_mtx(sd)
    norm = log1p_cpm_sparse(counts)

    Xspots, coverage = align_to_model_features(norm, ens_ids, model_features)
    if Xspots is None:
        raise RuntimeError(f"No overlap between model_features and Visium Ensembl IDs for {s}")

    rf_p = rf.predict_proba(Xspots)[:, 1]
    lr_p = lr.predict_proba(Xspots)[:, 1]

    out = pd.DataFrame({
        "barcode": spot_ids,
        "Sample": s,
        "Type": TYPE_MAP[s],
        "feature_coverage": coverage,
        "RF_pCAF": rf_p,
        "LR_pCAF": lr_p,
        "RF_pred": np.where(rf_p >= THR, "CAF_high", "Immune_high"),
        "LR_pred": np.where(lr_p >= THR, "CAF_high", "Immune_high"),
    })

    pos = read_tissue_positions(sd)
    out = out.merge(pos, on="barcode", how="left")

    # gene set scores (use SYMBOLS)
    for set_name, genes in GENESETS.items():
        out[f"{set_name}_score"] = score_geneset(norm, gene_symbols, genes)

    # save per-sample
    out_path = os.path.join(OUT_DIR, f"{s}_spots_predictions_with_genesets.csv")
    out.to_csv(out_path, index=False)

    all_spots.append(out)

    # per-sample gene-set summaries (median + fraction above threshold)
    # threshold defined as within-sample 75th percentile (transparent and scale-free)
    # You can change this to a fixed absolute threshold if you prefer.
    for set_name in GENESETS.keys():
        v = out[f"{set_name}_score"].values
        thr_q75 = np.nanquantile(v, 0.75)
        module_rows.append({
            "Sample": s,
            "Type": TYPE_MAP[s],
            "geneset": set_name,
            "median_score": float(np.nanmedian(v)),
            "q75_threshold": float(thr_q75),
            "frac_spots_above_q75": frac_above_threshold(v, thr_q75)
        })

spots = pd.concat(all_spots, ignore_index=True)
spots.to_csv(os.path.join(OUT_DIR, "ALL_spots_predictions_with_genesets.csv"), index=False)

module_summary = pd.DataFrame(module_rows)
module_summary.to_csv(os.path.join(OUT_DIR, "PerSample_GeneSetSummary_median_and_fracAboveQ75.csv"), index=False)

print("Saved spot predictions + per-sample gene set summaries.")


# ---------- 13) Per-sample ML summary (reviewer table) ----------
def iqr(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    return float(np.quantile(x, 0.75) - np.quantile(x, 0.25))

per_sample_ml = (
    spots.groupby(["Sample","Type"])
    .agg(
        n_spots=("barcode","size"),
        RF_frac_CAF=("RF_pred", lambda s: float((s=="CAF_high").mean())),
        LR_frac_CAF=("LR_pred", lambda s: float((s=="CAF_high").mean())),
        RF_median_p=("RF_pCAF","median"),
        LR_median_p=("LR_pCAF","median"),
        RF_iqr_p=("RF_pCAF", iqr),
        LR_iqr_p=("LR_pCAF", iqr),
        feature_coverage=("feature_coverage","median")
    )
    .reset_index()
    .sort_values(["Type","Sample"])
)
per_sample_ml.to_csv(os.path.join(OUT_DIR, "PerSample_ML_summary_RF_LR.csv"), index=False)
per_sample_ml


# ---------- 14) LOSO robustness on fraction CAF-high (RF + LR) ----------
def loso_effect(df, metric_col):
    samples = df["Sample"].unique().tolist()
    rows = []
    for left_out in samples:
        d = df[df["Sample"] != left_out]
        ocs = d.loc[d["Type"]=="OCS", metric_col].median()
        hgsc = d.loc[d["Type"]=="HGSC", metric_col].median()
        rows.append({
            "left_out": left_out,
            "metric": metric_col,
            "median_OCS": float(ocs),
            "median_HGSC": float(hgsc),
            "diff_OCS_minus_HGSC": float(ocs - hgsc)
        })
    return pd.DataFrame(rows)

loso = pd.concat([
    loso_effect(per_sample_ml, "RF_frac_CAF"),
    loso_effect(per_sample_ml, "LR_frac_CAF")
], ignore_index=True)

loso.to_csv(os.path.join(OUT_DIR, "LOSO_RF_LR_fracCAF.csv"), index=False)
loso


# ---------- 15) Spatial hub genes (2.11.7) ----------
# Step: correlate model feature gene expression with pCAF per sample,
# filter to model-derived hub candidates, recurrence >=2 samples within each tumor type,
# compute mean logCPM expression per sample and per type.

def get_expr_for_genes(norm_csr, visium_ens_ids, genes_ensembl):
    visium_clean = [g.split(".")[0] for g in visium_ens_ids]
    gene_to_idx = {g:i for i,g in enumerate(visium_clean)}

    genes_clean = [g.split(".")[0] for g in genes_ensembl]
    present = [g for g in genes_clean if g in gene_to_idx]
    idx = [gene_to_idx[g] for g in present]

    if len(idx) == 0:
        return None, []

    sub = norm_csr[idx, :]           # genes x spots
    return np.array(sub.todense()), present


def corr_table_for_sample(sample_dir, sample_name, model_features, rf, lr):
    counts, ens_ids, gene_symbols, spot_ids = read_visium_mtx(sample_dir)
    norm = log1p_cpm_sparse(counts)

    Xspots, _ = align_to_model_features(norm, ens_ids, model_features)
    rf_p = rf.predict_proba(Xspots)[:, 1]
    lr_p = lr.predict_proba(Xspots)[:, 1]

    expr, present_genes = get_expr_for_genes(norm, ens_ids, model_features)
    if expr is None:
        return None

    rows = []
    for i, g in enumerate(present_genes):
        x = expr[i, :]
        r_rf, p_rf = spearmanr(x, rf_p)
        r_lr, p_lr = spearmanr(x, lr_p)
        rows.append([sample_name, TYPE_MAP[sample_name], g, r_rf, p_rf, r_lr, p_lr])

    # IMPORTANT: keep the column name as 'gene' to match downstream code
    dfc = pd.DataFrame(rows, columns=["Sample","Type","gene","rho_RF","p_RF","rho_LR","p_LR"])
    dfc["abs_rho_RF"] = dfc["rho_RF"].abs()
    dfc["abs_rho_LR"] = dfc["rho_LR"].abs()
    return dfc

corr_all = []
for s in SAMPLES:
    sd = os.path.join(SAMPLES_DIR, s)
    dfc = corr_table_for_sample(sd, s, model_features, rf, lr)
    if dfc is None:
        raise RuntimeError(f"No overlap for hub correlation table in {s}")
    corr_all.append(dfc)

spatial_corr_all = pd.concat(corr_all, ignore_index=True)
spatial_corr_all.to_csv(os.path.join(OUT_DIR, "Spatial_modelFeature_vs_pCAF_correlations_all.csv"), index=False)

# filter to model-derived candidates (Ensembl IDs)
cand_clean = set([g.split(".")[0] for g in model_hub_candidates])
spatial_corr_filt = spatial_corr_all[spatial_corr_all["gene"].isin(cand_clean)].copy()

# recurrence >=2 samples within tumor type
recurrence = (
    spatial_corr_filt
    .groupby(["Type","gene"])["Sample"]
    .nunique()
    .reset_index(name="n_samples")
)
recurrent_genes = recurrence[recurrence["n_samples"] >= 2]

spatial_corr_recurrent = spatial_corr_filt.merge(
    recurrent_genes[["Type","gene"]],
    on=["Type","gene"],
    how="inner"
)

spatial_corr_recurrent.to_csv(os.path.join(OUT_DIR, "Spatial_hubs_recurrent_by_type.csv"), index=False)

print("Recurrent hub genes retained:", spatial_corr_recurrent["gene"].nunique())

# mean expression per sample for retained genes
def mean_expr_for_genes_in_sample(sample_dir, genes):
    # read 4 outputs now
    counts, ens_ids, gene_symbols, spot_ids = read_visium_mtx(sample_dir)
    norm = log1p_cpm_sparse(counts)

    ens_clean = [g.split(".")[0] for g in ens_ids]
    gene_to_idx = {}
    for i,g in enumerate(ens_clean):
        if g not in gene_to_idx:
            gene_to_idx[g] = i

    rows = []
    for g in genes:
        g0 = str(g).split(".")[0]
        if g0 not in gene_to_idx:
            continue
        v = norm[gene_to_idx[g0], :]
        rows.append([g0, float(np.array(v.mean()).ravel()[0])])

    return pd.DataFrame(rows, columns=["gene","mean_logCPM"])

expr_rows = []
for s in SAMPLES:
    sd = os.path.join(SAMPLES_DIR, s)
    genes_s = spatial_corr_recurrent.loc[spatial_corr_recurrent["Sample"] == s, "gene"].unique()
    if len(genes_s) == 0:
        continue
    df_expr = mean_expr_for_genes_in_sample(sd, genes_s)
    df_expr["Sample"] = s
    df_expr["Type"] = TYPE_MAP[s]
    expr_rows.append(df_expr)

expr_by_sample = pd.concat(expr_rows, ignore_index=True)
expr_by_sample.to_csv(os.path.join(OUT_DIR, "HubGenes_meanExpression_by_sample.csv"), index=False)

expr_by_type = (
    expr_by_sample
    .groupby(["Type","gene"])
    .agg(mean_logCPM=("mean_logCPM","mean"))
    .reset_index()
)
expr_by_type.to_csv(os.path.join(OUT_DIR, "HubGenes_meanExpression_by_type.csv"), index=False)

# ranked type-level hub tables (median abs corr)
rf_type_hubs = (
    spatial_corr_recurrent
    .groupby(["Type","gene"])
    .agg(
        n_samples=("Sample","nunique"),
        median_abs_rho_RF=("abs_rho_RF","median"),
        median_rho_RF=("rho_RF","median")
    )
    .reset_index()
    .sort_values(["Type","median_abs_rho_RF"], ascending=[True, False])
)
lr_type_hubs = (
    spatial_corr_recurrent
    .groupby(["Type","gene"])
    .agg(
        n_samples=("Sample","nunique"),
        median_abs_rho_LR=("abs_rho_LR","median"),
        median_rho_LR=("rho_LR","median")
    )
    .reset_index()
    .sort_values(["Type","median_abs_rho_LR"], ascending=[True, False])
)

rf_type_hubs.to_csv(os.path.join(OUT_DIR, "TypeLevel_hubs_RF_ranked.csv"), index=False)
lr_type_hubs.to_csv(os.path.join(OUT_DIR, "TypeLevel_hubs_LR_ranked.csv"), index=False)

print("Saved hub gene outputs.")


# ---------- 16) Correlation check (addresses 'not only stromal content') ----------
# Correlate pCAF with gene set scores per sample (Spearman).
corr_rows = []
score_cols = [c for c in spots.columns if c.endswith("_score")]

for (s, t), d in spots.groupby(["Sample","Type"]):
    for sc in score_cols:
        x_rf = d["RF_pCAF"].values
        x_lr = d["LR_pCAF"].values
        y_sc = d[sc].values

        ok = np.isfinite(x_rf) & np.isfinite(y_sc)
        if ok.sum() >= 10:
            r_rf, p_rf = spearmanr(x_rf[ok], y_sc[ok])
        else:
            r_rf, p_rf = np.nan, np.nan

        ok = np.isfinite(x_lr) & np.isfinite(y_sc)
        if ok.sum() >= 10:
            r_lr, p_lr = spearmanr(x_lr[ok], y_sc[ok])
        else:
            r_lr, p_lr = np.nan, np.nan

        corr_rows.append({"Sample": s, "Type": t, "model":"RF", "score": sc, "rho": r_rf, "p": p_rf})
        corr_rows.append({"Sample": s, "Type": t, "model":"LR", "score": sc, "rho": r_lr, "p": p_lr})

pCAF_vs_scores = pd.DataFrame(corr_rows).sort_values(["model","score","Type","Sample"])
pCAF_vs_scores.to_csv(os.path.join(OUT_DIR, "Correlation_pCAF_vs_GeneSetScores_RF_LR.csv"), index=False)

print("Saved pCAF vs gene-set correlation table.")


# ---------- 17) Optional spatial pCAF plots ----------
PLOT_DIR = os.path.join(OUT_DIR, "spot_level_plots")
os.makedirs(PLOT_DIR, exist_ok=True)

def plot_spatial_prob(df, sample_name, prob_col, out_png, cmap="turbo"):
    d = df.dropna(subset=["pxl_col","pxl_row"]).copy()
    plt.figure(figsize=(7,6))
    sc = plt.scatter(d["pxl_col"], d["pxl_row"], c=d[prob_col], s=6, cmap=cmap)
    plt.gca().invert_yaxis()
    plt.gca().set_aspect("equal", adjustable="box")
    plt.title(f"{sample_name} {prob_col}")
    plt.xticks([]); plt.yticks([])
    cbar = plt.colorbar(sc)
    cbar.set_label(prob_col)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

for s in SAMPLES:
    df_s = pd.read_csv(os.path.join(OUT_DIR, f"{s}_spots_predictions_with_genesets.csv"))
    plot_spatial_prob(df_s, s, "RF_pCAF", os.path.join(PLOT_DIR, f"{s}_RF_pCAF.png"))
    plot_spatial_prob(df_s, s, "LR_pCAF", os.path.join(PLOT_DIR, f"{s}_LR_pCAF.png"))

print("\nDONE. All outputs written to:", OUT_DIR)

In [ ]:

# =========================
# 13B) PSEUDO-SAMPLE EXPANSION FOR EXTERNAL VALIDATION
# Adds 20 bootstrap pseudo-samples from the 8 external Visium cases.
# IMPORTANT: these are NOT independent new patients. They are resampled
# spot-level pseudo-samples and should be reported as sensitivity/supporting
# analyses only, not as replacement for the true n=8 external cohort.
# =========================

# total number of pseudo-samples you want across the external validation cohort
N_PSEUDO_TOTAL = 20
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

# distribute 20 pseudo-samples across the 8 original samples as evenly as possible
sample_names = sorted(spots["Sample"].unique().tolist())
base_n = N_PSEUDO_TOTAL // len(sample_names)
remainder = N_PSEUDO_TOTAL % len(sample_names)

pseudo_counts = {
    s: base_n + (1 if i < remainder else 0)
    for i, s in enumerate(sample_names)
}
print("Pseudo-samples per original sample:", pseudo_counts)

pseudo_rows = []
pseudo_module_rows = []

score_cols = [c for c in spots.columns if c.endswith("_score")]

for s in sample_names:
    d0 = spots.loc[spots["Sample"] == s].copy().reset_index(drop=True)
    if d0.empty:
        continue

    sample_type = d0["Type"].iloc[0]
    n_spots = d0.shape[0]

    for k in range(pseudo_counts[s]):
        pseudo_id = f"{s}_pseudo_{k+1:02d}"

        # bootstrap spots WITH replacement from this original sample
        take_idx = rng.choice(n_spots, size=n_spots, replace=True)
        d = d0.iloc[take_idx].copy().reset_index(drop=True)

        # ---- pseudo-sample ML summary ----
        pseudo_rows.append({
            "PseudoSample": pseudo_id,
            "ParentSample": s,
            "Type": sample_type,
            "n_spots": int(d.shape[0]),
            "RF_frac_CAF": float((d["RF_pred"] == "CAF_high").mean()),
            "LR_frac_CAF": float((d["LR_pred"] == "CAF_high").mean()),
            "RF_median_p": float(np.nanmedian(d["RF_pCAF"].values)),
            "LR_median_p": float(np.nanmedian(d["LR_pCAF"].values)),
            "RF_iqr_p": iqr(d["RF_pCAF"].values),
            "LR_iqr_p": iqr(d["LR_pCAF"].values),
            "feature_coverage": float(np.nanmedian(d["feature_coverage"].values))
        })

        # ---- pseudo-sample gene-set summaries ----
        for sc in score_cols:
            v = d[sc].values.astype(float)
            thr_q75 = np.nanquantile(v, 0.75)

            pseudo_module_rows.append({
                "PseudoSample": pseudo_id,
                "ParentSample": s,
                "Type": sample_type,
                "geneset": sc.replace("_score", ""),
                "median_score": float(np.nanmedian(v)),
                "q75_threshold": float(thr_q75),
                "frac_spots_above_q75": frac_above_threshold(v, thr_q75)
            })

pseudo_external_ml = pd.DataFrame(pseudo_rows).sort_values(["Type", "ParentSample", "PseudoSample"])
pseudo_external_ml.to_csv(
    os.path.join(OUT_DIR, "PseudoExternalValidation_ML_summary_RF_LR.csv"),
    index=False
)

pseudo_external_modules = pd.DataFrame(pseudo_module_rows).sort_values(
    ["Type", "ParentSample", "PseudoSample", "geneset"]
)
pseudo_external_modules.to_csv(
    os.path.join(OUT_DIR, "PseudoExternalValidation_GeneSetSummary.csv"),
    index=False
)

# optional combined table: real samples + pseudo-samples
real_external_ml = per_sample_ml.copy()
real_external_ml["dataset_level"] = "real_external"
real_external_ml["sample_id"] = real_external_ml["Sample"]
real_external_ml["parent_sample"] = real_external_ml["Sample"]

pseudo_external_ml_combined = pseudo_external_ml.copy()
pseudo_external_ml_combined["dataset_level"] = "pseudo_external"
pseudo_external_ml_combined["sample_id"] = pseudo_external_ml_combined["PseudoSample"]
pseudo_external_ml_combined["parent_sample"] = pseudo_external_ml_combined["ParentSample"]

combined_external_ml = pd.concat([
    real_external_ml.rename(columns={"Sample": "original_sample"}),
    pseudo_external_ml_combined.rename(columns={"PseudoSample": "original_sample"})
], ignore_index=True, sort=False)

combined_external_ml.to_csv(
    os.path.join(OUT_DIR, "Combined_RealPlusPseudo_ExternalValidation_ML_summary_RF_LR.csv"),
    index=False
)

print("Saved:")
print(" - PseudoExternalValidation_ML_summary_RF_LR.csv")
print(" - PseudoExternalValidation_GeneSetSummary.csv")
print(" - Combined_RealPlusPseudo_ExternalValidation_ML_summary_RF_LR.csv")
print()
print("Real external samples:", per_sample_ml.shape[0])
print("Pseudo external samples:", pseudo_external_ml.shape[0])
print("Combined rows:", combined_external_ml.shape[0])

pseudo_external_ml.head()


In [ ]:
# =========================
# 18) EXPORT GUI ARTIFACTS (PKL + JSON)
# =========================
import os, json
import numpy as np
import joblib

ARTIFACT_DIR = os.path.join(OUT_DIR, "GUI_artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ---- preprocessing metadata (so the GUI knows what you did) ----
preprocessing = {
    "visium_input": "10x filtered_feature_bc_matrix (matrix.mtx + features.tsv + barcodes.tsv)",
    "gene_id_handling": "use Ensembl IDs; remove version suffix after '.'",
    "normalization": "CPM per spot then log1p",
    "feature_alignment": "align Visium genes to ordered model_features; fill missing with zeros",
    "model_input_shape": [None, len(model_features)],
    "threshold": THR
}

# ---- bundle RF + LR as pkl (model + feature order + preprocessing) ----
rf_bundle = {
    "name": "SOmics_RF",
    "model": rf,
    "model_features_ordered": model_features,
    "preprocessing": preprocessing
}
lr_bundle = {
    "name": "SOmics_LR_L1",
    "model": lr,
    "model_features_ordered": model_features,
    "preprocessing": preprocessing
}

rf_pkl = os.path.join(ARTIFACT_DIR, "somics_rf.pkl")
lr_pkl = os.path.join(ARTIFACT_DIR, "somics_lr.pkl")

joblib.dump(rf_bundle, rf_pkl)
joblib.dump(lr_bundle, lr_pkl)

print("Saved:", rf_pkl)
print("Saved:", lr_pkl)

# ---- hub genes JSON (exactly what the GUI will display) ----
# Use the already computed lists if available
try:
    lr_nonzero_list = sorted(list(lr_nonzero))
    rf_top_list = sorted(list(rf_top))
    hub_union_list = model_hub_candidates
except NameError:
    # fallback: recompute directly from models
    lr_coef_vec = lr.coef_.ravel()
    lr_nonzero_idx = np.where(lr_coef_vec != 0)[0]
    lr_nonzero_list = [model_features[i] for i in lr_nonzero_idx]

    rf_import = rf.feature_importances_
    topN = 200
    rf_top_idx = np.argsort(rf_import)[::-1][:topN]
    rf_top_list = [model_features[i] for i in rf_top_idx]

    hub_union_list = sorted(set(lr_nonzero_list).union(set(rf_top_list)))

# include weights/importances for ranking in the GUI
lr_coef = lr.coef_.ravel()
lr_weights = {model_features[i]: float(lr_coef[i]) for i in range(len(model_features)) if lr_coef[i] != 0}

rf_imp = rf.feature_importances_
rf_imps = {model_features[i]: float(rf_imp[i]) for i in np.argsort(rf_imp)[::-1][:200]}

hub_json = {
    "lr_nonzero_genes": lr_nonzero_list,
    "rf_top200_genes": rf_top_list,
    "hub_union_genes": hub_union_list,
    "lr_nonzero_weights": lr_weights,
    "rf_top200_importances": rf_imps
}

hub_path = os.path.join(ARTIFACT_DIR, "hub_genes.json")
with open(hub_path, "w") as f:
    json.dump(hub_json, f, indent=2)

print("Saved:", hub_path)

# ---- training params JSON (you already create params_out earlier) ----
# If params_out exists, save it again into the artifact folder for convenience
try:
    tp = params_out
except NameError:
    tp = {
        "rf_best_params": getattr(rf, "get_params", lambda: {})(),
        "lr_params": getattr(lr, "get_params", lambda: {})(),
        "threshold": THR,
        "n_features": int(len(model_features))
    }

train_params_path = os.path.join(ARTIFACT_DIR, "training_parameters.json")
with open(train_params_path, "w") as f:
    json.dump(tp, f, indent=2)

print("Saved:", train_params_path)

# ---- feature order JSON (extra safety) ----
feat_path = os.path.join(ARTIFACT_DIR, "model_features_1000.json")
with open(feat_path, "w") as f:
    json.dump({"model_features_ordered": model_features}, f, indent=2)

print("Saved:", feat_path)

print("\nDONE. GUI artifacts written to:", ARTIFACT_DIR)

In [ ]:
import joblib, os
p = os.path.join(OUT_DIR, "GUI_artifacts", "somics_rf.pkl")
bundle = joblib.load(p)
print(bundle.keys())
print("n_features:", len(bundle["model_features_ordered"]))

In [ ]:
# =========================
# ALL FIGURES FOR PI (FINAL)
# Uses your preferred spatial samples: OCS=Sample_2, HGSC=Sample_5
#
# Creates:
# Fig1  LOSO robustness (RF + LR)
# Fig2  Per-sample ML quantification: fraction CAF-high + IQR heterogeneity (RF + LR)
# Fig3  pCAF vs gene-set correlation dot-heatmaps (RF and LR)
# Fig4  Hub gene linkage:
#       (A) Top recurrent hubs per type (RF + LR barplots)
#       (B) Type-level mean expression heatmap for union of top hubs
# Fig5  Spatial maps panel (Sample_2 vs Sample_5; RF + LR)
#
# Inputs required (from your pipeline OUT_DIR):
#   LOSO_RF_LR_fracCAF.csv
#   PerSample_ML_summary_RF_LR.csv
#   Correlation_pCAF_vs_GeneSetScores_RF_LR.csv
#   TypeLevel_hubs_RF_ranked.csv
#   TypeLevel_hubs_LR_ranked.csv
#   HubGenes_meanExpression_by_type.csv
#   spot_level_plots/{Sample_X}_RF_pCAF.png and {Sample_X}_LR_pCAF.png
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


# ---- CONFIG ----
OUT_DIR = "/content/drive/MyDrive/ML_ExternalVal_FINAL"
FIG_DIR = os.path.join(OUT_DIR, "Figures_for_PI")
PLOT_DIR = os.path.join(OUT_DIR, "spot_level_plots")

OCS_SAMPLE  = "Sample_2"
HGSC_SAMPLE = "Sample_5"

TOPN_HUBS = 20  # per model per type

os.makedirs(FIG_DIR, exist_ok=True)

print("OUT_DIR:", OUT_DIR)
print("FIG_DIR:", FIG_DIR)
print("PLOT_DIR:", PLOT_DIR)
print("Spatial samples:", OCS_SAMPLE, "(OCS) vs", HGSC_SAMPLE, "(HGSC)")


# =========================
# helpers
# =========================
def assert_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()

def jitter(n, scale=0.06, seed=0):
    rng = np.random.default_rng(seed)
    return rng.normal(0, scale, size=n)


# =========================
# FIG 1 — LOSO robustness
# =========================
loso_path = os.path.join(OUT_DIR, "LOSO_RF_LR_fracCAF.csv")
assert_exists(loso_path)
loso = pd.read_csv(loso_path)

metrics = ["RF_frac_CAF", "LR_frac_CAF"]

fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for i, met in enumerate(metrics):
    d = loso[loso["metric"] == met].copy().sort_values("left_out")
    x = np.arange(len(d))
    y = d["diff_OCS_minus_HGSC"].values

    ax[i].axhline(0, linewidth=1)
    ax[i].scatter(x, y, s=50)
    ax[i].set_xticks(x)
    ax[i].set_xticklabels(d["left_out"].tolist(), rotation=45, ha="right")
    ax[i].set_title(f"LOSO robustness\n{met.replace('_', ' ')}")
    ax[i].set_ylabel("Δ median(OCS − HGSC)\n(fraction CAF-high)" if i == 0 else "")

savefig(os.path.join(FIG_DIR, "Fig1_LOSO_RF_LR_fracCAF.png"))
print("Saved Fig1")


# =========================
# FIG 2 — Per-sample ML quantification:
# (A) fraction CAF-high, (B) IQR of pCAF
# =========================
per_path = os.path.join(OUT_DIR, "PerSample_ML_summary_RF_LR.csv")
assert_exists(per_path)
per = pd.read_csv(per_path)

per["sample_num"] = per["Sample"].str.replace("Sample_", "", regex=False).astype(int)
per = per.sort_values("sample_num").copy()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# (A) fraction CAF-high
for model, col, seed in [("RF", "RF_frac_CAF", 1), ("LR", "LR_frac_CAF", 2)]:
    x0 = per["sample_num"].values.astype(float)
    x = x0 + jitter(len(x0), seed=seed)
    ax[0].scatter(x, per[col].values, s=50, label=model)

ax[0].axvline(4.5, linewidth=1)
ax[0].set_xticks(range(1, 9))
ax[0].set_xlabel("Sample # (1–4 OCS, 5–8 HGSC)")
ax[0].set_ylabel("Fraction CAF-high")
ax[0].set_title("Per-sample CAF-high fraction (binary)")
ax[0].legend()

# (B) IQR pCAF
for model, col, seed in [("RF", "RF_iqr_p", 11), ("LR", "LR_iqr_p", 12)]:
    x0 = per["sample_num"].values.astype(float)
    x = x0 + jitter(len(x0), seed=seed)
    ax[1].scatter(x, per[col].values, s=50, label=model)

ax[1].axvline(4.5, linewidth=1)
ax[1].set_xticks(range(1, 9))
ax[1].set_xlabel("Sample # (1–4 OCS, 5–8 HGSC)")
ax[1].set_ylabel("IQR of p(CAF-high)")
ax[1].set_title("Per-sample heterogeneity (IQR of probability)")
ax[1].legend()

savefig(os.path.join(FIG_DIR, "Fig2_PerSample_fraction_and_IQR.png"))
print("Saved Fig2")


# =========================
# FIG 3 — pCAF vs gene-set correlation dot-heatmaps (RF, LR)
# =========================
corr_path = os.path.join(OUT_DIR, "Correlation_pCAF_vs_GeneSetScores_RF_LR.csv")
assert_exists(corr_path)
corr = pd.read_csv(corr_path)

corr["score"] = corr["score"].str.replace("_score", "", regex=False)

gene_sets = list(dict.fromkeys(corr["score"].tolist()))  # preserve order
samples_order = [f"Sample_{i}" for i in range(1, 9)]

def dot_heatmap(df, title, outname):
    M = np.full((len(samples_order), len(gene_sets)), np.nan)
    for i, s in enumerate(samples_order):
        for j, g in enumerate(gene_sets):
            v = df[(df["Sample"] == s) & (df["score"] == g)]["rho"]
            if len(v) > 0:
                M[i, j] = float(v.iloc[0])

    plt.figure(figsize=(10, 4))
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M[i, j]):
                plt.scatter(j, i, s=180, c=[M[i, j]], vmin=-1, vmax=1)

    plt.colorbar(label="Spearman rho (pCAF vs gene-set score)")
    plt.yticks(range(len(samples_order)), samples_order)
    plt.xticks(range(len(gene_sets)), gene_sets, rotation=45, ha="right")
    plt.title(title)
    plt.gca().invert_yaxis()
    savefig(os.path.join(FIG_DIR, outname))

dot_heatmap(corr[corr["model"] == "RF"], "p(CAF-high) vs gene-set scores (RF)", "Fig3A_pCAF_vs_Genesets_RF.png")
dot_heatmap(corr[corr["model"] == "LR"], "p(CAF-high) vs gene-set scores (LR)", "Fig3B_pCAF_vs_Genesets_LR.png")
print("Saved Fig3A/Fig3B")


# =========================
# FIG 4 — Hub gene linkage
# (A) Top recurrent hubs per type (RF + LR)
# (B) Type-level expression heatmap for union of top hubs
# =========================
rf_hubs_path = os.path.join(OUT_DIR, "TypeLevel_hubs_RF_ranked.csv")
lr_hubs_path = os.path.join(OUT_DIR, "TypeLevel_hubs_LR_ranked.csv")
expr_type_path = os.path.join(OUT_DIR, "HubGenes_meanExpression_by_type.csv")

assert_exists(rf_hubs_path)
assert_exists(lr_hubs_path)
assert_exists(expr_type_path)

rf_hubs = pd.read_csv(rf_hubs_path)
lr_hubs = pd.read_csv(lr_hubs_path)
expr_type = pd.read_csv(expr_type_path)

# union of top hubs per type per model
top_rf = rf_hubs.groupby("Type").head(TOPN_HUBS)[["Type", "gene"]]
top_lr = lr_hubs.groupby("Type").head(TOPN_HUBS)[["Type", "gene"]]
top_genes = sorted(set(top_rf["gene"]).union(set(top_lr["gene"])))

# (A) barplots
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

def plot_topbars(df, type_name, abs_col, ax_here, title):
    d = df[df["Type"] == type_name].head(TOPN_HUBS).copy()
    ax_here.barh(d["gene"].astype(str), d[abs_col].values)
    ax_here.invert_yaxis()
    ax_here.set_title(title)
    ax_here.set_xlabel("Median |Spearman rho| across samples")
    ax_here.set_ylabel("Gene (Ensembl)")

plot_topbars(rf_hubs, "OCS",  "median_abs_rho_RF", ax[0,0], "RF hubs (OCS)")
plot_topbars(rf_hubs, "HGSC", "median_abs_rho_RF", ax[0,1], "RF hubs (HGSC)")
plot_topbars(lr_hubs, "OCS",  "median_abs_rho_LR", ax[1,0], "LR hubs (OCS)")
plot_topbars(lr_hubs, "HGSC", "median_abs_rho_LR", ax[1,1], "LR hubs (HGSC)")

savefig(os.path.join(FIG_DIR, "Fig4A_TopRecurrentHubs_RF_LR_byType.png"))
print("Saved Fig4A")

# (B) expression heatmap by type (OCS/HGSC)
expr_sub = expr_type[expr_type["gene"].isin(top_genes)].copy()
mat = expr_sub.pivot_table(index="gene", columns="Type", values="mean_logCPM", aggfunc="mean")

for col in ["OCS", "HGSC"]:
    if col not in mat.columns:
        mat[col] = np.nan
mat = mat[["OCS", "HGSC"]]

# order genes by absolute type difference to make plot interpretable
mat["abs_diff"] = (mat["OCS"] - mat["HGSC"]).abs()
mat = mat.sort_values("abs_diff", ascending=False).drop(columns=["abs_diff"])

plt.figure(figsize=(6, max(6, 0.25 * mat.shape[0])))
plt.imshow(mat.values, aspect="auto")
plt.colorbar(label="Mean logCPM")
plt.yticks(range(mat.shape[0]), mat.index.astype(str))
plt.xticks([0,1], ["OCS", "HGSC"])
plt.title("Type-level mean expression of top recurrent hubs")
savefig(os.path.join(FIG_DIR, "Fig4B_HubGeneExpression_heatmap_byType.png"))
print("Saved Fig4B")


# =========================
# FIG 5 — Spatial pCAF maps panel (your chosen samples)
# =========================
rf1 = os.path.join(PLOT_DIR, f"{OCS_SAMPLE}_RF_pCAF.png")
lr1 = os.path.join(PLOT_DIR, f"{OCS_SAMPLE}_LR_pCAF.png")
rf2 = os.path.join(PLOT_DIR, f"{HGSC_SAMPLE}_RF_pCAF.png")
lr2 = os.path.join(PLOT_DIR, f"{HGSC_SAMPLE}_LR_pCAF.png")

for p in [rf1, lr1, rf2, lr2]:
    assert_exists(p)

imgs = [mpimg.imread(rf1), mpimg.imread(lr1), mpimg.imread(rf2), mpimg.imread(lr2)]
titles = [
    f"{OCS_SAMPLE} (OCS) RF p(CAF-high)",
    f"{OCS_SAMPLE} (OCS) LR p(CAF-high)",
    f"{HGSC_SAMPLE} (HGSC) RF p(CAF-high)",
    f"{HGSC_SAMPLE} (HGSC) LR p(CAF-high)",
]

fig, ax = plt.subplots(2, 2, figsize=(10, 8))
k = 0
for i in range(2):
    for j in range(2):
        ax[i, j].imshow(imgs[k])
        ax[i, j].set_title(titles[k])
        ax[i, j].axis("off")
        k += 1

savefig(os.path.join(FIG_DIR, "Fig5_Spatial_pCAF_maps_Sample2_vs_Sample5_RF_LR.png"))
print("Saved Fig5")


print("\nALL DONE. Figures saved here:\n", FIG_DIR)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load combined spot table if "spots" isn't in memory
all_spots_csv = os.path.join(OUT_DIR, "ALL_spots_predictions_with_genesets.csv")
if "spots" not in globals():
    spots = pd.read_csv(all_spots_csv)

print("spots shape:", spots.shape)
print("columns (preview):", [c for c in spots.columns if c.endswith("_score")][:8], "...")

In [ ]:
FIG_DIR = os.path.join(OUT_DIR, "paper_figures_new")
os.makedirs(FIG_DIR, exist_ok=True)

def iqr(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    return float(np.quantile(x, 0.75) - np.quantile(x, 0.25))

def plot_prob_distributions_by_type(spots, prob_col, out_png, title):
    plt.figure(figsize=(7.5, 4.5))

    types = ["OCS", "HGSC"]
    data = []
    for t in types:
        v = spots.loc[spots["Type"] == t, prob_col].values
        v = v[np.isfinite(v)]
        data.append(v)

    # Violin + points summary
    parts = plt.violinplot(data, showmeans=False, showmedians=False, showextrema=False)
    for i, t in enumerate(types, start=1):
        v = data[i-1]
        med = np.median(v)
        iq = iqr(v)
        plt.scatter([i], [med], s=60)  # median dot
        plt.text(i+0.06, med, f"median={med:.2f}\nIQR={iq:.2f}", va="center", fontsize=9)

    plt.xticks([1,2], types)
    plt.ylim(0, 1)
    plt.ylabel(prob_col)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.show()

plot_prob_distributions_by_type(
    spots, "RF_pCAF",
    os.path.join(FIG_DIR, "ProbDist_byType_RF_pCAF.png"),
    "Spot-level p(CAF-high) distributions by tumor type (RF)"
)

plot_prob_distributions_by_type(
    spots, "LR_pCAF",
    os.path.join(FIG_DIR, "ProbDist_byType_LR_pCAF.png"),
    "Spot-level p(CAF-high) distributions by tumor type (LR)"
)

In [ ]:
def plot_per_sample_histograms(spots, prob_col, out_png, title):
    samples = sorted(spots["Sample"].unique().tolist())
    n = len(samples)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    plt.figure(figsize=(14, 3.2*nrows))

    for i, s in enumerate(samples, start=1):
        plt.subplot(nrows, ncols, i)
        v = spots.loc[spots["Sample"] == s, prob_col].values
        v = v[np.isfinite(v)]
        plt.hist(v, bins=30)
        t = spots.loc[spots["Sample"] == s, "Type"].iloc[0]
        plt.title(f"{s} ({t})")
        plt.xlim(0,1)
        plt.xlabel(prob_col)
        plt.ylabel("spots")

    plt.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

plot_per_sample_histograms(
    spots, "RF_pCAF",
    os.path.join(FIG_DIR, "PerSample_hist_RF_pCAF.png"),
    "Per-sample spot-level p(CAF-high) distributions (RF)"
)

plot_per_sample_histograms(
    spots, "LR_pCAF",
    os.path.join(FIG_DIR, "PerSample_hist_LR_pCAF.png"),
    "Per-sample spot-level p(CAF-high) distributions (LR)"
)

In [ ]:
def scatter_pCAF_vs_geneset(spots, sample, prob_col, geneset_key, out_png=None, title=None):
    score_col = f"{geneset_key}_score"
    d = spots.loc[spots["Sample"] == sample, [prob_col, score_col]].dropna()
    x = d[score_col].values
    y = d[prob_col].values

    # Spearman correlation
    from scipy.stats import spearmanr
    r, p = spearmanr(x, y)

    plt.figure(figsize=(5.3, 4.2))
    plt.scatter(x, y, s=6, alpha=0.6)
    plt.xlabel(f"{geneset_key} score (spot-level)")
    plt.ylabel(prob_col)
    t = spots.loc[spots["Sample"] == sample, "Type"].iloc[0]
    plt.title(title or f"{sample} ({t}): {prob_col} vs {geneset_key}\nSpearman ρ={r:.2f}, p={p:.1e}")
    plt.ylim(0,1)
    plt.tight_layout()
    if out_png:
        plt.savefig(out_png, dpi=300)
    plt.show()

# Pick the key “suppression” sets you want to emphasize
KEY_SETS = ["stromal_caf_exclusion", "checkpoint_exhaustion", "myeloid_suppressive"]

for sample in ["Sample_2", "Sample_5"]:
    for prob_col in ["RF_pCAF", "LR_pCAF"]:
        for gs in KEY_SETS:
            scatter_pCAF_vs_geneset(
                spots, sample, prob_col, gs,
                out_png=os.path.join(FIG_DIR, f"{sample}_{prob_col}_scatter_vs_{gs}.png")
            )

In [ ]:
from sklearn.calibration import calibration_curve

def plot_reliability_curve(y_true, p_pred, out_png, title):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)

    prob_true, prob_pred = calibration_curve(y_true, p_pred, n_bins=10, strategy="quantile")

    plt.figure(figsize=(5.5, 5.2))
    plt.plot([0,1], [0,1], 'k--', alpha=0.6, label="Perfect calibration")
    plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label="Model")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed fraction CAF-high")
    plt.title(title)
    plt.ylim(0,1); plt.xlim(0,1)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.show()

plot_reliability_curve(
    y_test, rf_test_p,
    os.path.join(FIG_DIR, "Calibration_RF_TCGA_test.png"),
    "Reliability curve (RF) on held-out TCGA test set"
)

plot_reliability_curve(
    y_test, lr_test_p,
    os.path.join(FIG_DIR, "Calibration_LR_TCGA_test.png"),
    "Reliability curve (LR-L1) on held-out TCGA test set"
)

In [ ]:
from scipy.stats import spearmanr

def compute_symbol_expression_per_spot(sample_dir, target_symbols):
    counts, ens_ids, gene_symbols, spot_ids = read_visium_mtx(sample_dir)
    norm = log1p_cpm_sparse(counts)  # genes x spots

    sym = [s.split(".")[0] for s in gene_symbols]
    sym_to_idx = {}
    for i,g in enumerate(sym):
        if g not in sym_to_idx:
            sym_to_idx[g] = i

    out = {"barcode": spot_ids}
    for g in target_symbols:
        if g in sym_to_idx:
            v = np.array(norm[sym_to_idx[g], :].todense()).ravel()
            out[g] = v
        else:
            out[g] = np.nan
    return pd.DataFrame(out)

def fn1_col1a1_check(sample):
    sd = os.path.join(SAMPLES_DIR, sample)
    expr = compute_symbol_expression_per_spot(sd, ["FN1","COL1A1"])
    d = spots.loc[spots["Sample"] == sample, ["barcode","RF_pCAF","LR_pCAF",
                                              "stromal_caf_exclusion_score"]].merge(expr, on="barcode", how="left")
    # Correlations
    for model in ["RF_pCAF","LR_pCAF"]:
        for gene in ["FN1","COL1A1"]:
            ok = np.isfinite(d[model]) & np.isfinite(d[gene])
            r, p = spearmanr(d.loc[ok, gene], d.loc[ok, model])
            print(f"{sample} {model} vs {gene}: Spearman ρ={r:.2f}, p={p:.1e}")

    # Scatter plots
    for model in ["RF_pCAF","LR_pCAF"]:
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1)
        plt.scatter(d["FN1"], d[model], s=6, alpha=0.6)
        plt.xlabel("FN1 (log1p CPM)")
        plt.ylabel(model)
        plt.ylim(0,1)
        plt.title(f"{sample}: {model} vs FN1")

        plt.subplot(1,2,2)
        plt.scatter(d["COL1A1"], d[model], s=6, alpha=0.6)
        plt.xlabel("COL1A1 (log1p CPM)")
        plt.ylabel(model)
        plt.ylim(0,1)
        plt.title(f"{sample}: {model} vs COL1A1")

        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, f"{sample}_{model}_vs_FN1_COL1A1.png"), dpi=300)
        plt.show()

# Run for your two representative samples
fn1_col1a1_check("Sample_2")
fn1_col1a1_check("Sample_5")

In [ ]:
def partial_corr(x, y, z):
    # residualize x and y on z using simple linear regression, then correlate residuals
    x = np.asarray(x); y = np.asarray(y); z = np.asarray(z)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x = x[ok]; y = y[ok]; z = z[ok]
    if len(x) < 20:
        return np.nan

    X = np.column_stack([np.ones_like(z), z])
    bx, *_ = np.linalg.lstsq(X, x, rcond=None)
    by, *_ = np.linalg.lstsq(X, y, rcond=None)

    rx = x - X @ bx
    ry = y - X @ by
    r, _ = spearmanr(rx, ry)
    return float(r)

def partial_corr_demo(sample, model="RF_pCAF"):
    d = spots.loc[spots["Sample"] == sample].copy()
    strom = d["stromal_caf_exclusion_score"].values
    chk   = d["checkpoint_exhaustion_score"].values
    myel  = d["myeloid_suppressive_score"].values
    pCAF  = d[model].values

    r_chk = partial_corr(pCAF, chk, strom)
    r_my  = partial_corr(pCAF, myel, strom)
    print(f"{sample} ({model}): partial corr(pCAF, checkpoint | stromal) = {r_chk:.2f}")
    print(f"{sample} ({model}): partial corr(pCAF, myeloid | stromal)   = {r_my:.2f}")

partial_corr_demo("Sample_2", "RF_pCAF")
partial_corr_demo("Sample_2", "LR_pCAF")
partial_corr_demo("Sample_5", "RF_pCAF")
partial_corr_demo("Sample_5", "LR_pCAF")

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score

# Held-out test predictions from the original SOmics models
y_true = np.asarray(y_test).astype(int)
predictions = {
    "Random Forest": np.asarray(rf_test_p),
    "Logistic Regression": np.asarray(lr_test_p)
}

N_BOOTSTRAPS = 5000
rng = np.random.default_rng(42)

# Stratified bootstrap: resample patients within each class
class_0 = np.where(y_true == 0)[0]
class_1 = np.where(y_true == 1)[0]

bootstrap_indices = []
for _ in range(N_BOOTSTRAPS):
    idx = np.concatenate([
        rng.choice(class_0, size=len(class_0), replace=True),
        rng.choice(class_1, size=len(class_1), replace=True)
    ])
    bootstrap_indices.append(idx)

results = []

for model_name, probabilities in predictions.items():
    auc_original = roc_auc_score(y_true, probabilities)
    acc_original = accuracy_score(
        y_true, (probabilities >= THR).astype(int)
    )

    auc_values = []
    acc_values = []

    for idx in bootstrap_indices:
        y_boot = y_true[idx]
        p_boot = probabilities[idx]

        auc_values.append(roc_auc_score(y_boot, p_boot))
        acc_values.append(
            accuracy_score(y_boot, (p_boot >= THR).astype(int))
        )

    auc_ci = np.percentile(auc_values, [2.5, 97.5])
    acc_ci = np.percentile(acc_values, [2.5, 97.5])

    results.append({
        "Model": model_name,
        "AUC": round(auc_original, 3),
        "AUC 95% CI lower": round(auc_ci[0], 3),
        "AUC 95% CI upper": round(auc_ci[1], 3),
        "Accuracy": round(acc_original, 3),
        "Accuracy 95% CI lower": round(acc_ci[0], 3),
        "Accuracy 95% CI upper": round(acc_ci[1], 3)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Save for manuscript supplementary materials
results_df.to_csv(
    "/content/drive/MyDrive/ML Train Data/TCGA_Bootstrap_95CI.csv",
    index=False
)
print("\nSaved bootstrap confidence intervals.")


In [ ]:

import os
import numpy as np
import pandas as pd

# Location of your original SOmics results
OUT_DIR = "/content/drive/MyDrive/ML_ExternalVal_FINAL"

# Load original tissue-level summary
file_path = os.path.join(
    OUT_DIR, "PerSample_ML_summary_RF_LR.csv"
)

df = pd.read_csv(file_path)

# Verify that the dataset contains 4 OCS and 4 HGSC samples
print("Original samples by tumor type:")
print(df.groupby("Type")["Sample"].nunique())

assert len(df) == 8, "Expected exactly 8 original samples."
assert df["Sample"].is_unique, "Samples must be unique."
assert df.groupby("Type")["Sample"].nunique().to_dict() == {
    "OCS": 4, "HGSC": 4
}, "Expected 4 OCS and 4 HGSC samples."

# Bootstrap at the original tissue-sample level
N_BOOT = 10000
rng = np.random.default_rng(42)

results = []

for model, col in [
    ("Random Forest", "RF_frac_CAF"),
    ("Logistic Regression", "LR_frac_CAF")
]:
    ocs = df.loc[df["Type"] == "OCS", col].to_numpy()
    hgsc = df.loc[df["Type"] == "HGSC", col].to_numpy()

    # Observed difference in mean CAF-high fractions
    observed = ocs.mean() - hgsc.mean()

    # Resample original tissues, NOT individual spots
    boot_diffs = []

    for i in range(N_BOOT):
        ocs_boot = rng.choice(
            ocs, size=len(ocs), replace=True
        )
        hgsc_boot = rng.choice(
            hgsc, size=len(hgsc), replace=True
        )

        boot_diffs.append(
            ocs_boot.mean() - hgsc_boot.mean()
        )

    ci_low, ci_high = np.percentile(
        boot_diffs, [2.5, 97.5]
    )

    results.append({
        "Model": model,
        "OCS_mean_CAF_fraction": ocs.mean(),
        "HGSC_mean_CAF_fraction": hgsc.mean(),
        "OCS_minus_HGSC": observed,
        "Bootstrap_CI_lower": ci_low,
        "Bootstrap_CI_upper": ci_high
    })

results_df = pd.DataFrame(results)

print("\nTISSUE-LEVEL BOOTSTRAP RESULTS:")
print(results_df.round(4).to_string(index=False))

results_df.to_csv(
    os.path.join(
        OUT_DIR, "Tissue_Level_Bootstrap_95CI.csv"
    ),
    index=False
)

print("\nSaved tissue-level bootstrap results.")


In [ ]:

import os
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/ML_ExternalVal_FINAL"

# Load existing leave-one-sample-out results
loso_file = os.path.join(
    OUT_DIR, "LOSO_RF_LR_fracCAF.csv"
)

loso = pd.read_csv(loso_file)

print("Columns:", loso.columns.tolist())
print("\nLeave-one-sample-out results:")
print(loso.to_string(index=False))

print("\nSaved results loaded successfully.")

In [ ]:

import os
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/ML_ExternalVal_FINAL"

print("Files related to pseudo-sampling:\n")

for root, dirs, files in os.walk(OUT_DIR):
    for filename in sorted(files):
        if any(term in filename.lower() for term in
               ["pseudo", "bootstrap", "robust", "figure16"]):
            path = os.path.join(root, filename)
            print("\nFILE:", path)

            if filename.lower().endswith(".csv"):
                try:
                    data = pd.read_csv(path)
                    print("Shape:", data.shape)
                    print("Columns:", data.columns.tolist())
                    print(data.head(3).to_string(index=False))
                except Exception as e:
                    print("Could not read:", e)


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Use the original trained models and feature alignment
required = [
    "rf", "lr", "model_features", "SAMPLES_DIR",
    "SAMPLES", "TYPE_MAP", "read_visium_mtx",
    "log1p_cpm_sparse", "align_to_model_features"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        "Run the original setup/training cell first. Missing: "
        + ", ".join(missing)
    )

SAVE_DIR = "/content/drive/MyDrive/ML_ExternalVal_FINAL/Normalization_Sensitivity"
os.makedirs(SAVE_DIR, exist_ok=True)

rows = []
spot_results = []

for sample in SAMPLES:
    print("Processing:", sample)

    sample_dir = os.path.join(SAMPLES_DIR, sample)
    counts, ens_ids, symbols, barcodes = read_visium_mtx(sample_dir)

    # Original: natural log(CPM + 1)
    norm_original = log1p_cpm_sparse(counts)

    # Alternative: log2(CPM + 1)
    norm_log2 = norm_original.copy()
    norm_log2.data = norm_log2.data / np.log(2)

    X_original, coverage_1 = align_to_model_features(
        norm_original, ens_ids, model_features
    )
    X_log2, coverage_2 = align_to_model_features(
        norm_log2, ens_ids, model_features
    )

    if X_original is None or X_log2 is None:
        raise RuntimeError(f"Feature alignment failed: {sample}")

    for model_name, model in [
        ("Random Forest", rf),
        ("Logistic Regression", lr)
    ]:
        p_original = model.predict_proba(X_original)[:, 1]
        p_log2 = model.predict_proba(X_log2)[:, 1]

        threshold = 0.5

        frac_original = np.mean(p_original >= threshold)
        frac_log2 = np.mean(p_log2 >= threshold)

        rho = spearmanr(p_original, p_log2).statistic
        agreement = np.mean(
            (p_original >= threshold) ==
            (p_log2 >= threshold)
        )

        rows.append({
            "Sample": sample,
            "Type": TYPE_MAP[sample],
            "Model": model_name,
            "n_spots": len(barcodes),
            "Original_CAF_fraction": frac_original,
            "Log2_CAF_fraction": frac_log2,
            "Change": frac_log2 - frac_original,
            "Original_median_pCAF": np.median(p_original),
            "Log2_median_pCAF": np.median(p_log2),
            "Spearman_rho": rho,
            "Classification_agreement": agreement
        })

        spot_results.append(pd.DataFrame({
            "Sample": sample,
            "Type": TYPE_MAP[sample],
            "barcode": barcodes,
            "Model": model_name,
            "Original_pCAF": p_original,
            "Log2_pCAF": p_log2
        }))

results = pd.DataFrame(rows)

print("\nPER-SAMPLE NORMALIZATION SENSITIVITY")
print(results.round(4).to_string(index=False))

print("\nTUMOR-TYPE MEAN CAF-HIGH FRACTIONS")
summary = (
    results.groupby(["Model", "Type"])[
        ["Original_CAF_fraction", "Log2_CAF_fraction"]
    ]
    .mean()
)
print(summary.round(4).to_string())

print("\nOCS MINUS HGSC DIFFERENCES")
for model_name in ["Random Forest", "Logistic Regression"]:
    s = summary.loc[model_name]
    original_diff = (
        s.loc["OCS", "Original_CAF_fraction"]
        - s.loc["HGSC", "Original_CAF_fraction"]
    )
    log2_diff = (
        s.loc["OCS", "Log2_CAF_fraction"]
        - s.loc["HGSC", "Log2_CAF_fraction"]
    )
    print(model_name)
    print("  Original difference:", round(original_diff, 4))
    print("  Log2 difference:    ", round(log2_diff, 4))
    print("  Change in difference:", round(log2_diff-original_diff, 4))

results.to_csv(
    os.path.join(SAVE_DIR, "Normalization_PerSample_Comparison.csv"),
    index=False
)

pd.concat(spot_results, ignore_index=True).to_csv(
    os.path.join(SAVE_DIR, "Normalization_SpotPredictions.csv"),
    index=False
)

print("\nSaved results to:", SAVE_DIR)


In [ ]:

import numpy as np
import pandas as pd
import os

print("=== MODEL AND TRAINING DATA CHECK ===")

for name in ["rf", "lr", "X_train_red", "model_features"]:
    if name in globals():
        print(f"{name}: Available")
    else:
        print(f"{name}: MISSING")

if all(name in globals() for name in
       ["rf", "lr", "X_train_red", "model_features"]):

    print("\nTraining data shape:", X_train_red.shape)
    print("Number of model features:", len(model_features))

    # Confirm feature names and exact ordering
    for name, model in [("RF", rf), ("LR", lr)]:
        saved_features = getattr(
            model, "feature_names_in_", None
        )

        if saved_features is not None:
            matched = np.array_equal(
                np.asarray(saved_features).astype(str),
                np.asarray(model_features).astype(str)
            )
            print(f"{name} feature order matches:", matched)
        else:
            print(f"{name}: No saved feature names")

    # Describe original TCGA input distribution
    arr = X_train_red.to_numpy(dtype=float)

    print("\n=== TCGA TRAINING VALUE DISTRIBUTION ===")
    print("Minimum:", np.nanmin(arr))
    print("Median:", np.nanmedian(arr))
    print("95th percentile:", np.nanpercentile(arr, 95))
    print("Maximum:", np.nanmax(arr))
    print("Negative values:", np.sum(arr < 0))
    print("Non-finite values:", np.sum(~np.isfinite(arr)))

print("\n=== SAVED NORMALIZATION RESULTS ===")

path = (
    "/content/drive/MyDrive/ML_ExternalVal_FINAL/"
    "Normalization_Sensitivity/"
    "Normalization_PerSample_Comparison.csv"
)

if os.path.exists(path):
    d = pd.read_csv(path)
    print("Saved comparison rows:", len(d))
    print("Samples:", d["Sample"].nunique())
    print("Models:", d["Model"].unique().tolist())
else:
    print("Saved comparison file not found.")


In [ ]:

import numpy as np
import pandas as pd
import os
from scipy import sparse

def summarize_values(X, name):
    values = X.toarray() if sparse.issparse(X) else np.asarray(X)
    values = values.ravel()

    return {
        "Dataset": name,
        "Minimum": np.min(values),
        "Median": np.median(values),
        "95th_percentile": np.percentile(values, 95),
        "Maximum": np.max(values),
        "Fraction_zero": np.mean(values == 0)
    }

summaries = [
    summarize_values(X_train_red, "TCGA training")
]

for sample in SAMPLES:
    sample_dir = os.path.join(SAMPLES_DIR, sample)

    counts, ens_ids, symbols, barcodes = read_visium_mtx(
        sample_dir
    )

    norm = log1p_cpm_sparse(counts)

    X_spatial, coverage = align_to_model_features(
        norm, ens_ids, model_features
    )

    summaries.append(
        summarize_values(X_spatial, sample)
    )

distribution_df = pd.DataFrame(summaries)

print(distribution_df.round(4).to_string(index=False))


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

rows = []

for sample in SAMPLES:
    sample_dir = os.path.join(SAMPLES_DIR, sample)

    counts, ens_ids, symbols, barcodes = read_visium_mtx(
        sample_dir
    )

    norm = log1p_cpm_sparse(counts)

    X_spatial, coverage = align_to_model_features(
        norm, ens_ids, model_features
    )

    # Fraction of detected model genes in each spot
    detection_fraction = (
        X_spatial.getnnz(axis=1) / X_spatial.shape[1]
        if hasattr(X_spatial, "getnnz")
        else np.mean(np.asarray(X_spatial) > 0, axis=1)
    )

    for model_name, model in [
        ("Random Forest", rf),
        ("Logistic Regression", lr)
    ]:
        p_caf = model.predict_proba(X_spatial)[:, 1]

        rho, p_value = spearmanr(
            detection_fraction, p_caf
        )

        rows.append({
            "Sample": sample,
            "Type": TYPE_MAP[sample],
            "Model": model_name,
            "Median_detection_fraction": np.median(
                detection_fraction
            ),
            "Spearman_rho": rho,
            "P_value": p_value
        })

results = pd.DataFrame(rows)

print("\nSPARSITY VS CAF-HIGH PREDICTION")
print(results.round(4).to_string(index=False))

print("\nSUMMARY BY MODEL")
print(
    results.groupby("Model")["Spearman_rho"]
    .agg(["mean", "min", "max"])
    .round(4)
)

save_dir = (
    "/content/drive/MyDrive/ML_ExternalVal_FINAL/"
    "Normalization_Sensitivity"
)

os.makedirs(save_dir, exist_ok=True)

results.to_csv(
    os.path.join(
        save_dir,
        "Sparsity_vs_CAF_Probability.csv"
    ),
    index=False
)
